# Yiginlanmis Topluluk (CatBoost + LightGBM + RF) - Risk Skorlamasi

**Katman:** Guclendirici

Literaturde deprem buyuklugu/risk kestiriminde en iyi sonucu veren kurgu. Meta-ogrenici: lojistik regresyon. Ayni walk-forward protokolu.

In [1]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/05_risk_skoru
    %pip install -q catboost xgboost lightgbm shap scikit-learn
print('cwd:', os.getcwd())

cwd: /home/utku/Desktop/QuakeMindLD/QuakeMind/model_benchmarks/05_risk_skoru


## 1. Veri

In [2]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd, common_preprocessing as cp
X, y, df_main = cp.load_dataset()
splits = cp.walk_forward_splits(len(X), 5)

## 2. Yiginlanmis model

In [3]:
MODEL_NAME = "stacking_ensemble"
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
PARAMS = dict(final_estimator='LogisticRegression', cv=3)
def build():
    return StackingClassifier(
        estimators=[('cat', CatBoostClassifier(iterations=600, depth=7, learning_rate=0.03, verbose=False, random_seed=42)),
                    ('lgbm', LGBMClassifier(n_estimators=600, learning_rate=0.03, random_state=42, verbose=-1)),
                    ('rf', RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42, n_jobs=-1))],
        final_estimator=LogisticRegression(max_iter=1000), cv=3, n_jobs=-1)

## 3. Walk-forward

In [4]:
fold_metrics, oof_t, oof_p = [], [], []
for i,(tr,te) in enumerate(splits,1):
    m = build(); m.fit(X.iloc[tr], y.iloc[tr])
    prob = m.predict_proba(X.iloc[te])[:,1]
    fm = cp.evaluate_predictions(y.iloc[te], prob); fold_metrics.append(fm)
    oof_t.append(np.asarray(y.iloc[te])); oof_p.append(prob)
    print(f"fold {i}: ROC-AUC={fm['roc_auc']:.3f} PR-AUC={fm['pr_auc']:.3f} F1={fm['f1']:.3f}")
agg = cp.aggregate_folds(fold_metrics)
lb = cp.save_result(MODEL_NAME, agg, extra={'params': PARAMS})
lb[['model','roc_auc_mean','pr_auc_mean','f1_mean','brier_mean']]

fold 1: ROC-AUC=0.227 PR-AUC=0.901 F1=0.127


fold 2: ROC-AUC=0.979 PR-AUC=0.998 F1=0.951


fold 3: ROC-AUC=0.918 PR-AUC=0.960 F1=0.855


fold 4: ROC-AUC=0.807 PR-AUC=0.866 F1=0.782


fold 5: ROC-AUC=0.738 PR-AUC=0.640 F1=0.598


,model,roc_auc_mean,pr_auc_mean,f1_mean,brier_mean
0,catboost,0.846470,0.901110,0.852457,0.136769
4,random_forest,0.841747,0.905813,0.857216,0.131835
6,xgboost,0.795654,0.892862,0.856252,0.136286
2,logistic_regression,0.777560,0.866459,0.811794,0.151116
5,stacking_ensemble,0.733814,0.872902,0.662508,0.225919
1,lightgbm,0.716763,0.876670,0.827409,0.172441
3,mlp,0.427598,0.710838,0.674202,0.228834
